# Saddle Transport — Framework

Consolidated framework from the research log in [saddle_transport.ipynb](saddle_transport.ipynb). This notebook imports the stable kinematics primitives from `miscope.analysis.saddle_transport` and applies them to the current variant set from fresh artifacts.

## Framework

The **saddle-transport picture** proposes that training dynamics pass through a saddle region whose axis is rotated from the PCs of the parameter trajectory. Transit along this axis is sigmoidal. When multiple saddle-like events occur in sequence, the trajectory looks more like a **heteroclinic chain / tube-dynamics** traversal: each valley-bounded segment contributes its own local transit axis.

## Two-axis taxonomy

Per-variant behaviour lives on a plane spanned by:

- **Sigmoidality** (mean Δ = R²_sig − R²_lin, averaged across valley-bounded segments) — how saddle-like individual transitions are.
- **Coupling** (mean |cos θ| across pairs of per-segment transit axes) — how aligned successive transits are.

High coupling + low Δ = chord-linear (single-axis fast). Low coupling + high Δ = serial-chain (multiple independent sigmoidal axes = tube-dynamics fit). Coupling is a **kinematic** property, not a health axis.

## What moved to the module

`src/miscope/analysis/saddle_transport.py` — pure library functions operating on existing `parameter_trajectory` and `parameter_snapshot` artifacts. No plotting, no rule-based classifier (both stay in-notebook).


## 1. Setup

In [13]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import find_peaks

from miscope import load_family
from miscope.analysis import saddle_transport as st

family = load_family('modulo_addition_1layer')

## 2. Variant inventory

Seven variants in scope. Existing-class labels carry forward from the research log; new variants await classification.

| # | variant | prior class | dense coverage |
|---|---|---|---|
| 1 | p109/s485/ds598 | chord-linear (fast-rigid) | **refreshed — fully dense incl. late** |
| 2 | p113/s999/ds598 | two-saddle distinct (canon) | dense through 25k |
| 3 | p101/s485/ds999 | two-saddle continuous (rebounder) | **dense through 25k** |
| 4 | p59/s485/ds598 | overshoot-then-tumble | dense through 35k |
| 5 | p101/s999/ds598 | serial-chain (long-bumpy) | dense through 35k |
| 6 | **p101/s485/ds598** | new — pending | **dense through 25K** |
| 7 | **p113/s485/ds999** | new — 2nd rebounder candidate | **dense through 25K** |


In [ ]:
VARIANT_SPECS = [
    dict(prime=109, seed=485, data_seed=598, prior_class='chord-linear'),
    dict(prime=113, seed=999, data_seed=598, prior_class='two-saddle distinct (canon)'),
    dict(prime=101, seed=485, data_seed=999, prior_class='two-saddle continuous (rebounder)'),
    dict(prime=59,  seed=485, data_seed=598, prior_class='overshoot-then-tumble'),
    dict(prime=101, seed=999, data_seed=598, prior_class='serial-chain (long-bumpy)'),
    dict(prime=101, seed=485, data_seed=598, prior_class='NEW'),
    dict(prime=113, seed=485, data_seed=999, prior_class='NEW (rebounder candidate)'),
    dict(prime=89, seed=999, data_seed=999, prior_class='NEW'),
]

## 3. Run pipeline — fresh artifacts

`run_saddle_pipeline` loads MLP params, detects wiggle (PR₃ peak in `[fd_end, grok or sd_onset]`), computes transit projection + ds/dt peak. No caching.

In [15]:
RUNS = {}
for spec in VARIANT_SPECS:
    variant = family.get_variant(prime=spec['prime'], seed=spec['seed'], data_seed=spec['data_seed'])
    try:
        R = st.run_saddle_pipeline(variant)
    except Exception as e:
        print(f"p{spec['prime']}/s{spec['seed']}/ds{spec['data_seed']}: {type(e).__name__}: {e}")
        continue
    R['prior_class'] = spec['prior_class']
    RUNS[R['label']] = R
    print(f"{R['label']:22s}  n={len(R['epochs']):3d}  d_mlp={R['X'].shape[1]}  "
          f"wiggle={R['wiggle_pr3']:>5d}  saddle={R['saddle']:>5d}  L={R['L']:6.2f}  "
          f"class={spec['prior_class']}")

p109/s485/ds598         n=251  d_mlp=131072  wiggle= 1700  saddle= 1500  L= 30.90  class=chord-linear
p113/s999/ds598         n=251  d_mlp=131072  wiggle= 6800  saddle=10400  L= 27.62  class=two-saddle distinct (canon)
p101/s485/ds999         n=251  d_mlp=131072  wiggle= 5500  saddle= 7200  L= 24.66  class=two-saddle continuous (rebounder)
p59/s485/ds598          n=352  d_mlp=131072  wiggle= 1700  saddle= 1400  L= 27.14  class=overshoot-then-tumble
p101/s999/ds598         n=352  d_mlp=131072  wiggle=10000  saddle=14000  L= 26.29  class=serial-chain (long-bumpy)
p101/s485/ds598         n=251  d_mlp=131072  wiggle= 2000  saddle= 1500  L= 32.35  class=NEW
p113/s485/ds999         n=251  d_mlp=131072  wiggle= 2200  saddle= 1500  L= 33.43  class=NEW (rebounder candidate)


## 4. Valley topology + arc decomposition + valley-bounded segments

For each variant:

1. `compute_arc_decomposition` — local-PC1 vs arc-tangent alignment along the `mlp__projections` trajectory.
2. Peaks in rolling PR₃ at prominence 0.05 in the `[fd_end, commit]` window.
3. `analyze_valley_topology` — inter-peak depth / reach-to-one / intervening maxes.
4. `build_valley_segments` — one segment per prominent peak, bounded by adjacent valleys.

In [16]:
DECOMP = {}
TOPOLOGY = {}
SEGMENTS = {}
for label, R in RUNS.items():
    decomp = st.compute_arc_decomposition(R['mlp_proj'], R['pt_eps'])
    DECOMP[label] = decomp

    fd_end = R['timing']['fd_end']
    commit = st.commitment_epoch(R['timing'])
    in_win = (R['pt_eps'] >= fd_end) & (R['pt_eps'] <= commit)
    pr3_win = R['pr3'][in_win]
    eps_win = R['pt_eps'][in_win]
    peaks_idx, _ = find_peaks(pr3_win, prominence=0.05)

    topo = st.analyze_valley_topology(pr3_win, eps_win, peaks_idx)
    TOPOLOGY[label] = topo
    SEGMENTS[label] = st.build_valley_segments(R['pr3'], R['pt_eps'], fd_end, commit, topo)

    peak_eps = [int(eps_win[i]) for i in peaks_idx]
    print(f"{label:22s}  peaks={peak_eps}  n_segs={len(SEGMENTS[label])}")

p109/s485/ds598         peaks=[1700, 3400]  n_segs=2
p113/s999/ds598         peaks=[2000, 6800]  n_segs=2
p101/s485/ds999         peaks=[1800, 5500]  n_segs=2
p59/s485/ds598          peaks=[1700, 2600, 9400, 16100, 22100]  n_segs=5
p101/s999/ds598         peaks=[2000, 10000, 17700]  n_segs=3
p101/s485/ds598         peaks=[2000, 8800]  n_segs=2
p113/s485/ds999         peaks=[2200, 10000]  n_segs=2


## 5. Local sigmoid fits per segment

`fit_local_sigmoid` builds a local transit axis from segment endpoints in full MLP parameter space, then fits sigmoid-vs-linear. **Sigmoidality Δ = R²_sig − R²_lin**: Δ≥0.05 = saddle-like, 0.02–0.05 = mildly sigmoidal, <0.02 = drift-like.

In [17]:
LOCAL_SIGMOID = {}
print(f"{'variant':22s}  {'seg':>3s}  {'window':>18s}  {'peak':>6s}  {'L':>7s}  "
      f"{'R²_sig':>7s}  {'R²_lin':>7s}  {'Δ':>7s}  {'align':>6s}")
print('-' * 110)
for label, R in RUNS.items():
    segs = SEGMENTS[label]
    fd_end = R['timing']['fd_end']
    commit = st.commitment_epoch(R['timing'])
    in_win = (R['pt_eps'] >= fd_end) & (R['pt_eps'] <= commit)
    align_win = DECOMP[label]['alignment'][in_win]
    eps_win = R['pt_eps'][in_win]

    entries = []
    for seg in segs:
        fit = st.fit_local_sigmoid(R['X'], R['epochs'], seg['ep_lo'], seg['ep_hi'])
        if fit is None:
            continue
        pk_rel = int(np.argmin(np.abs(eps_win - seg['peak_ep'])))
        peak_align = float(align_win[pk_rel])
        flavor = 'on' if peak_align > 0.7 else ('off' if peak_align < 0.4 else 'mix')
        entry = dict(
            segment_idx=seg['segment_idx'], peak_ep=seg['peak_ep'],
            ep_lo=seg['ep_lo'], ep_hi=seg['ep_hi'],
            L=fit['L'], r2_sig=fit['r2_sig'], r2_lin=fit['r2_lin'],
            sigmoidality=fit['sigmoidality'],
            align=peak_align, flavor=flavor,
        )
        entries.append(entry)
        print(f"{label:22s}  {seg['segment_idx']:>3d}  [{seg['ep_lo']:>5d},{seg['ep_hi']:>6d}]  "
              f"{seg['peak_ep']:>6d}  {fit['L']:>7.2f}  {fit['r2_sig']:>7.3f}  "
              f"{fit['r2_lin']:>7.3f}  {fit['sigmoidality']:>+7.3f}  {peak_align:>6.2f}")
    LOCAL_SIGMOID[label] = entries

variant                 seg              window    peak        L   R²_sig   R²_lin        Δ   align
--------------------------------------------------------------------------------------------------------------
p109/s485/ds598           0  [ 1200,  2200]    1700    12.48    0.999    0.994   +0.005    0.42
p109/s485/ds598           1  [ 2200,  5243]    3400    21.65    0.999    0.997   +0.003    0.80
p113/s999/ds598           0  [ 1200,  4400]    2000    20.94    0.996    0.959   +0.037    0.13
p113/s999/ds598           1  [ 4400, 12448]    6800    27.55    0.997    0.958   +0.039    0.93
p101/s485/ds999           0  [ 1200,  3700]    1800    18.49    0.996    0.967   +0.030    0.28
p101/s485/ds999           1  [ 3700,  9300]    5500    24.37    0.999    0.975   +0.024    0.77
p59/s485/ds598            0  [ 1000,  2100]    1700    12.89    1.000    0.990   +0.010    0.33
p59/s485/ds598            1  [ 2100,  5800]    2600    12.36    0.998    0.980   +0.018    0.94
p59/s485/ds598       

## 6. Local axes + pairwise geometry

`extract_local_axes` pulls a unit transit vector `v̂` per segment in full MLP parameter space. `pairwise_angle_matrix` gives the sign-flip-invariant angle between segment axes — **low angles (high |cos|) = coupled transits; high angles near 90° = orthogonal transits.**

In [18]:
LOCAL_AXES = {}
ANGLES = {}
for label, R in RUNS.items():
    axes = st.extract_local_axes(LOCAL_SIGMOID[label], R['epochs'], R['X'])
    LOCAL_AXES[label] = axes
    if len(axes) >= 2:
        M = st.pairwise_angle_matrix(axes)
        ANGLES[label] = M
        off = M[np.triu_indices_from(M, k=1)]
        cos_vals = np.cos(np.radians(off))
        print(f"{label:22s}  n={len(axes)}  median={np.median(off):5.1f}°  "
              f"min={off.min():5.1f}°  max={off.max():5.1f}°  mean|cos|={np.mean(cos_vals):.3f}")
    else:
        ANGLES[label] = None
        print(f"{label:22s}  n={len(axes)}  (single axis — no pairs)")

p109/s485/ds598         n=2  median= 54.3°  min= 54.3°  max= 54.3°  mean|cos|=0.583
p113/s999/ds598         n=2  median= 77.4°  min= 77.4°  max= 77.4°  mean|cos|=0.218
p101/s485/ds999         n=2  median= 71.9°  min= 71.9°  max= 71.9°  mean|cos|=0.311
p59/s485/ds598          n=5  median= 82.5°  min= 44.8°  max= 89.0°  mean|cos|=0.178
p101/s999/ds598         n=3  median= 87.4°  min= 81.5°  max= 89.4°  mean|cos|=0.068
p101/s485/ds598         n=2  median= 79.8°  min= 79.8°  max= 79.8°  mean|cos|=0.177
p113/s485/ds999         n=2  median= 83.6°  min= 83.6°  max= 83.6°  mean|cos|=0.112


## 7. Variant summary — (coupling × sigmoidality) plane

In [19]:
rows = []
for label, R in RUNS.items():
    axes = LOCAL_AXES[label]
    if not axes:
        continue
    mean_delta = float(np.mean([a['sigmoidality'] for a in axes]))
    mean_vel = float(np.mean([a['velocity'] for a in axes]))
    if ANGLES[label] is not None:
        off = ANGLES[label][np.triu_indices_from(ANGLES[label], k=1)]
        mean_cos = float(np.mean(np.cos(np.radians(off))))
        median_ang = float(np.median(off))
    else:
        mean_cos, median_ang = np.nan, np.nan
    rows.append(dict(
        variant=label, prior_class=R['prior_class'], n_segs=len(axes),
        mean_delta=mean_delta, mean_vel=mean_vel,
        mean_abs_cos=mean_cos, median_angle=median_ang,
    ))
SUMMARY = pd.DataFrame(rows).set_index('variant')
SUMMARY.round(4)

,prior_class,n_segs,mean_delta,mean_vel,mean_abs_cos,median_angle
variant,,,,,,
p109/s485/ds598,chord-linear,2,0.0040,0.0098,0.5834,54.3070
p113/s999/ds598,two-saddle distinct (canon),2,0.0379,0.0050,0.2181,77.4054
p101/s485/ds999,two-saddle continuous (rebounder),2,0.0270,0.0059,0.3108,71.8950
p59/s485/ds598,overshoot-then-tumble,5,0.0114,0.0040,0.1776,82.4521
p101/s999/ds598,serial-chain (long-bumpy),3,0.0597,0.0028,0.0678,87.4349
p101/s485/ds598,NEW,2,0.0658,0.0038,0.1772,79.7960
p113/s485/ds999,NEW (rebounder candidate),2,0.0687,0.0034,0.1121,83.5616


## 8. Figure — Δ vs velocity per segment

Sigmoidality × chord velocity, each point a segment. Colour = variant; symbol = on/off/mix arc-flavour.

In [20]:
palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
flavor_symbol = {'on': 'circle', 'off': 'x', 'mix': 'diamond'}

fig = go.Figure()
for i, (label, R) in enumerate(RUNS.items()):
    color = palette[i % len(palette)]
    for axis in LOCAL_AXES[label]:
        fig.add_trace(go.Scatter(
            x=[axis['velocity']], y=[axis['sigmoidality']],
            mode='markers', showlegend=axis['segment_idx'] == 0,
            name=label,
            marker=dict(color=color, size=12, symbol=flavor_symbol.get(axis['flavor'], 'circle'),
                        line=dict(color='black', width=1)),
            text=f"seg{axis['segment_idx']} @ ep{axis['peak_ep']} ({axis['flavor']})",
            hovertemplate='%{text}<br>Δ=%{y:.3f}<br>velocity=%{x:.4f}<extra>' + label + '</extra>',
        ))
fig.add_hline(y=0.05, line=dict(color='gray', dash='dot', width=1),
              annotation_text='Δ=0.05 (saddle-like)', annotation_position='right')
fig.add_hline(y=0.02, line=dict(color='gray', dash='dot', width=1),
              annotation_text='Δ=0.02 (mild)', annotation_position='right')
fig.update_layout(
    title='Sigmoidality Δ vs chord velocity — per-segment',
    xaxis_title='velocity (L / Δepoch)', yaxis_title='Δ = R²_sig − R²_lin',
    template='plotly_white', height=500, width=900,
)
fig.show()

## 9. Figure — coupling × sigmoidality (primary)

The 2-axis framework plot. **X** = mean |cos| across pairs (1.0 = all axes parallel, 0.0 = all axes orthogonal). **Y** = mean Δ across segments. Annotate each point with the variant's prior class.

In [21]:
df = SUMMARY.dropna(subset=['mean_abs_cos'])
fig = go.Figure()
for i, (label, row) in enumerate(df.iterrows()):
    color = palette[i % len(palette)]
    fig.add_trace(go.Scatter(
        x=[row['mean_abs_cos']], y=[row['mean_delta']],
        mode='markers+text', name=label,
        text=[f"{label}<br>{row['prior_class']}"], textposition='top center',
        textfont=dict(size=10),
        marker=dict(color=color, size=16, line=dict(color='black', width=1)),
        hovertemplate=f"{label}<br>n_segs={int(row['n_segs'])}<br>"
                      '|cos|=%{x:.3f}<br>Δ=%{y:.3f}<extra></extra>',
    ))

single_axis = SUMMARY[SUMMARY['mean_abs_cos'].isna()]
for label, row in single_axis.iterrows():
    fig.add_annotation(x=0.5, y=row['mean_delta'], text=f"{label} (1 seg)",
                       showarrow=False, font=dict(size=10, color='gray'))

fig.update_layout(
    title='Regime plane — coupling × sigmoidality',
    xaxis=dict(title='mean |cos θ|  ←— orthogonal   parallel —→', range=[-0.05, 1.0]),
    yaxis=dict(title='mean Δ  (mean sigmoidality)'),
    template='plotly_white', height=800, width=900, showlegend=False,
)
fig.show()

## 10. Figure — pairwise-angle heatmap per variant

Direct view of the coupling structure. Diagonals are 0° by construction. Bright off-diagonal = orthogonal axes (serial chain); dark off-diagonal = aligned axes (coupled / single-direction).

In [22]:
multi_seg_labels = [lbl for lbl in RUNS if ANGLES.get(lbl) is not None]
if multi_seg_labels:
    cols = min(len(multi_seg_labels), 4)
    rows_n = (len(multi_seg_labels) + cols - 1) // cols
    fig = make_subplots(rows=rows_n, cols=cols,
                        subplot_titles=multi_seg_labels, horizontal_spacing=0.08)
    for idx, label in enumerate(multi_seg_labels):
        r, c = idx // cols + 1, idx % cols + 1
        M = ANGLES[label]
        fig.add_trace(go.Heatmap(
            z=M, colorscale='Viridis', zmin=0, zmax=90,
            showscale=(idx == 0),
            colorbar=dict(title='angle (deg)', len=0.9) if idx == 0 else None,
            text=np.round(M, 1), texttemplate='%{text}', textfont=dict(size=10),
        ), row=r, col=c)
    fig.update_layout(title='Pairwise segment-axis angles (|cos|-based, sign-flip invariant)',
                      template='plotly_white', height=300 * rows_n, width=280 * cols + 100)
    fig.show()
else:
    print('No multi-segment variants in current set.')

## 11. Regime taxonomy (from research log)

Reference anchors from the research log — each regime is an empirical cluster on the (coupling × sigmoidality) plane.

| regime | mean \|cos\| | mean Δ | example | reading |
|---|---|---|---|---|
| chord-linear | ~0.60 | ~0.003 | p109/s485/ds598 | single-axis fast traversal |
| two-saddle continuous | ~0.37 | ~0.025 | p101/s485/ds999 | two saddles share direction |
| two-saddle distinct | ~0.22 | ~0.038 | p113/s999/ds598 canon | two independent saddles |
| serial-chain | ~0.07 | ~0.060 | p101/s999/ds598 | multiple orthogonal sigmoidal axes |
| overshoot-then-tumble | mixed | mixed | p59/s485/ds598 | coupled pair early, near-null later |

Null baseline (high-D): in d_mlp ≈ O(10⁵), expected |cos| ≈ 1/√d ≈ 0.003, angle ≈ 90°. Observed |cos| ≥ 0.01 is above chance.

**Coupling is not a health axis** — p109 and canon both grok and sit at opposite coupling extremes.

## 12. Next experiments

Four open questions, prioritized over further descriptive evidence.

1. **What is the transit axis mechanistically?** Geometric ≠ functional. §18 sketch (per-freq-group centroid kinks) is the first concrete bridge to circuit structure — does a global PR₃ peak coincide with multi-group kinks (collective saddle) or single-group kinks (staggered chain)?
2. **Causal falsification via parameter-space perturbation.** Each segment's `v̂` from `extract_local_axes` is a ready-to-use perturbation direction — perturb at wiggle along `v̂`, observe whether the model recovers via a different route (axis not load-bearing) or permanently derails (axis load-bearing).
3. **Universality.** The 2L MLP family shows no geometric arc. Either this picture is attention-bearing-architecture-specific, or the 2L MLP has a hidden arc we don't know how to see.
4. **Cross-site coupling.** Attn↔MLP handoff (§16 of the research log) may be the missing dimension that makes tube-dynamics self-consistent. Stuck-orbit pathology (p59) could be a site that failed to hand off.

### Open questions this fresh run may resolve

- **p109 late-training signatures.** Now that p109 is dense through end-of-training, does the late-training behaviour change the regime classification, or sit outside `[fd_end, commit]` and leave the framework signature unchanged?
- **p101/s485/ds598 baseline.** First dense analysis of this p101 variant. Where does it sit in the (coupling × sigmoidality) plane? Particularly interesting given the p101 family is anomaly-prone.
- **p113/s485/ds999 as 2nd rebounder.** Sharp test — does `max(s) > 1.0` under `terminal=sd_end` replicate the p101/s485/ds999 rebounder signature, or is rebounder-ness more idiosyncratic?